### Level 3: Data Quality Checks

In [1]:
import pandas as pd

In [2]:
#load dataset
df=pd.read_excel(r"C:\Users\Dell\Dataset1.csv.xlsx")
df.head()

,SN,Train_No,Station_Code,1A,2A,3A,SL,Station_Name,Route_Number,Arrival_time,Departure_Time,Distance
0,1,107,SWV,100,100,100,100,SAWANTWADI R,1,00:00:00,10:25:00,0
1,2,107,THVM,260,228,196,164,THIVIM,1,11:06:00,11:08:00,32
2,3,107,KRMI,345,296,247,198,KARMALI,1,11:28:00,11:30:00,49
3,4,107,MAO,490,412,334,256,MADGOAN JN.,1,12:10:00,00:00:00,78
4,1,108,MAO,100,100,100,100,MADGOAN JN.,1,00:00:00,20:30:00,0


In [3]:
#Task 3.1 — Handle Missing Schedule Values
df.isnull().sum()

SN                0
Train_No          0
Station_Code      0
1A                0
2A                0
3A                0
SL                0
Station_Name      0
Route_Number      0
Arrival_time      0
Departure_Time    0
Distance          0
dtype: int64

In [5]:
#Check the dataset size
df.shape

(186074, 12)

In [6]:
# Check empty or whitespace-only strings in each column
for col in df.select_dtypes(include='object').columns:
    blank_count = df[col].astype('string').str.strip().eq('').sum()
    print(f"{col}: {blank_count} blank values")

Station_Code: 0 blank values
Station_Name: 0 blank values
Arrival_time: 0 blank values
Departure_Time: 0 blank values


### Task 3.1 — Handle Missing Schedule Values

**Objective:** Check the dataset for missing or blank values and ensure the schedule data is complete.

**Steps performed:**
- Checked missing values in all columns using `df.isnull().sum()`.
- Inspected text columns for empty or whitespace-only strings.
- Verified the dataset dimensions.

**Results:**
- Total records: 186,074
- Total columns: 12
- Missing values: 0
- Blank values in checked text columns: 0

**Conclusion:**

The dataset contained no missing values or blank strings in the checked text columns. Therefore, no missing-value imputation or deletion was required for this task.

In [7]:
#Task 3.2 — Remove Duplicate Train Records.
df.duplicated().sum()

np.int64(0)

In [9]:
#Check the train count
df['Train_No'].nunique()

11113

In [8]:
#Check duplicate train–station combinations
df.duplicated(subset=['Train_No', 'Station_Code']).sum()

np.int64(30)

In [10]:
# Display rows with repeated train and station-code combinations
duplicate_records = df[df.duplicated(subset=['Train_No', 'Station_Code'],
        keep=False)].sort_values(by=['Train_No', 'Station_Code'])

duplicate_records

,SN,Train_No,Station_Code,1A,2A,3A,SL,Station_Name,Route_Number,Arrival_time,Departure_Time,Distance
30,1,290,DSJ,100,100,100,100,DELHI-SAFDAR,1,18:30:00,18:30:00,0
43,14,290,DSJ,13570,10876,8182,5488,DELHI-SAFDAR,1,02:30:00,02:30:00,2694
68,3,477,BNW,805,664,523,382,BHIWANI JN.,1,15:55:00,16:20:00,141
77,12,477,BNW,12470,9996,7522,5048,BHIWANI JN.,1,11:05:00,11:35:00,2474
71,6,477,CNB,3910,3148,2386,1624,KANPUR CENTR,1,05:30:00,05:35:00,762
74,9,477,CNB,9350,7500,5650,3800,KANPUR CENTR,1,21:00:00,21:05:00,1850
70,5,477,DSJ,1685,1368,1051,734,DELHI-SAFDAR,1,20:13:00,20:15:00,317
75,10,477,DSJ,11595,9296,6997,4698,DELHI-SAFDAR,1,07:13:00,07:15:00,2299
67,2,477,HSR,505,424,343,262,HISAR JN,1,13:55:00,14:25:00,81
78,13,477,HSR,12770,10236,7702,5168,HISAR JN,1,13:10:00,13:40:00,2534


In [11]:
## Count exact duplicate rows
df.duplicated().sum()

np.int64(0)

### Task 3.2 — Remove Duplicate Train Records

**Objective:** Identify duplicate records and prevent the removal of valid train schedule entries.

**Steps performed:**
- Checked for exact duplicate rows using `df.duplicated().sum()`.
- Checked repeated combinations of `Train_No` and `Station_Code`.
- Inspected repeated combinations to distinguish genuine duplicates from valid station visits.

**Results:**
- Exact duplicate rows: 0
- Repeated train–station-code combinations: 30

**Conclusion:**
No exact duplicate rows were found. The repeated train–station-code combinations included records with different distances, indicating that some trains may visit the same station more than once. Therefore, no rows were deleted based solely on repeated train–station-code combinations.

In [12]:
#Task 3.3 — Verify Station Order

# Display station order for a sample train
df[df['Train_No'] == 107][['SN', 'Train_No', 'Station_Code', 'Station_Name',
     'Route_Number', 'Distance']].head(20)

,SN,Train_No,Station_Code,Station_Name,Route_Number,Distance
0,1,107,SWV,SAWANTWADI R,1,0
1,2,107,THVM,THIVIM,1,32
2,3,107,KRMI,KARMALI,1,49
3,4,107,MAO,MADGOAN JN.,1,78


In [14]:
# Sort records by train and serial number
df_sorted = df.sort_values(['Train_No', 'SN']).copy()

# Compare each station's distance with the previous station
df_sorted['Previous_Distance'] = (
    df_sorted.groupby('Train_No')['Distance'].shift(1))

In [15]:
# Find records where distance decreases
distance_decreases = df_sorted[
    df_sorted['Distance'] < df_sorted['Previous_Distance']]

distance_decreases[['Train_No', 'SN', 'Station_Name',
    'Previous_Distance', 'Distance']].head(20)

,Train_No,SN,Station_Name,Previous_Distance,Distance


In [16]:
# Count records where distance decreases
distance_decreases.shape[0]

0

In [17]:
# Check how many records were examined
df_sorted.shape

(186074, 13)

In [18]:
#Check the station sequence
df_sorted[df_sorted['Train_No'] == 107][
    ['SN', 'Station_Code', 'Station_Name', 'Distance']].head(15)

,SN,Station_Code,Station_Name,Distance
0,1,SWV,SAWANTWADI R,0
1,2,THVM,THIVIM,32
2,3,KRMI,KARMALI,49
3,4,MAO,MADGOAN JN.,78


### Task 3.3 — Verify Station Order

#### Objective
To verify the station sequence for each train by examining serial numbers and comparing cumulative distances between consecutive stations.

#### Steps Performed
- Sorted the dataset by `Train_No` and `SN`.
- Used the `groupby()` and `shift()` methods to compare each station's distance with the previous station.
- Identified records where cumulative distance decreased.
- Inspected the station sequence for Train 107 to verify the distance progression.

#### Results
- **Total records examined:** 186,074
- **Distance decreases detected:** 0
- **Sample train inspected:** Train 107
- **Observed distance progression:** 0 km → 32 km → 49 km → 78 km

#### Conclusion
No decreasing cumulative distances were detected across the examined records when sorted by train number and serial number. The inspected sample also followed the expected distance progression. This check did not identify any distance-order issues; however, it does not independently verify the geographical correctness of every train's route.

In [19]:
#Task 3.4: Save the Verified Dataset!

# Create a copy of the dataset for final validation
df_final = df.copy()

# Check the dataset dimensions
df_final.shape

(186074, 12)

In [20]:
#Verify the dataset
# Check missing values
df_final.isnull().sum().sum()

np.int64(0)

In [21]:
# Check exact duplicate rows
df_final.duplicated().sum()

np.int64(0)

In [22]:
# Save the verified dataset
df_final.to_csv('train_schedule_verified.csv',index=False)